# Zypher — Train Your Chatbot From Scratch (Google Colab)

This notebook clones the repo from GitHub and trains the **custom Transformer** (tokenizer → pretrain → SFT → chat).

**Before you start:** Runtime → Change runtime type → **T4 GPU** (or better).

In [ ]:
# Check GPU
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    raise RuntimeError('No GPU detected. Go to Runtime → Change runtime type → GPU')

In [ ]:
# Clone the GitHub repo (nothing stored on your PC)
REPO = 'https://github.com/FOUNDEROF-AIRIES-AGENT/Zypher-Training-data.git'
BRANCH = 'cursor/llm-finetune-ready-da1e'  # change to 'main' after merge

!rm -rf Zypher-Training-data
!git clone --branch {BRANCH} --single-branch {REPO}
%cd Zypher-Training-data

In [ ]:
# Install dependencies
!pip install -q -r requirements.txt

In [ ]:
# Validate setup before training (re-run after each major step)
!python3 scripts/validate_pipeline.py

## Step 1 — Prepare training data

`generate-smoke` creates ~2k synthetic files (fast). For more data later, increase `--max-files` or run full `make generate`.

In [ ]:
!make generate-smoke
!make prepare-advanced
!cat data/advanced/dataset_stats.json

## Step 2 — Train BPE tokenizer

In [ ]:
!python3 -m chatbot.cli train-tokenizer \
  --corpus-glob "knowledge-base/**/*.md" \
  --output outputs/tokenizer

## Step 3 — Pretrain Transformer (from random weights)

Adjust `--max-steps` for longer training. Colab T4: use `--batch-size 2` if you hit OOM.

In [ ]:
PRETRAIN_STEPS = 5000   # increase to 50000 for a stronger model
BATCH_SIZE = 1          # use 1 on Colab T4 to avoid OOM

!python3 -m chatbot.cli pretrain \
  --data data/advanced/pretrain.txt \
  --tokenizer outputs/tokenizer \
  --output outputs/pretrain \
  --batch-size {BATCH_SIZE} \
  --max-steps {PRETRAIN_STEPS}

In [ ]:
!python3 scripts/validate_pipeline.py --require-checkpoint pretrain

In [ ]:
# Verify pretrain checkpoint exists BEFORE running SFT
from pathlib import Path

final_ckpt = Path("outputs/pretrain/checkpoint-final/model.pt")
if final_ckpt.exists():
    print("OK: pretrain checkpoint found at", final_ckpt)
else:
    print("ERROR: pretrain did not finish. Available checkpoints:")
    !ls -la outputs/pretrain/ 2>/dev/null || echo "outputs/pretrain/ missing — re-run pretrain cell above"
    raise FileNotFoundError(
        "Pretrain checkpoint missing. Re-run the pretrain cell and wait for "
        "'Saved checkpoint to outputs/pretrain/checkpoint-final'"
    )

## Step 4 — Supervised fine-tuning (chat behavior)

In [ ]:
!python3 scripts/validate_pipeline.py --require-checkpoint sft

In [ ]:
SFT_STEPS = 5000   # increase to 30000 for a stronger model

!python3 -m chatbot.cli sft \
  --data data/advanced/train.jsonl \
  --tokenizer outputs/tokenizer \
  --checkpoint outputs/pretrain/checkpoint-final \
  --output outputs/sft \
  --batch-size 1 \
  --max-steps {SFT_STEPS} \
  --lr 2e-5

## Step 5 — Chat with your model

In [ ]:
!python3 -m chatbot.cli chat --checkpoint outputs/sft/checkpoint-final

## Step 6 — Download checkpoints (before Colab session ends)

Colab wipes files when the session ends. Download your trained model here.

In [ ]:
!zip -r zypher-from-scratch.zip outputs/sft/checkpoint-final outputs/tokenizer

from google.colab import files
files.download('zypher-from-scratch.zip')

## Optional — one-cell shortcut

Runs a shorter smoke training (`make train-scratch`) if you just want to verify the pipeline.

In [ ]:
# Uncomment to run quick smoke test instead of full steps above:
# !make train-scratch